In [2]:
import torch.nn as nn
import torch

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads ==0), \
        "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads ##Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj = nn.Linear(d_out,d_out) #Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length),diagonal=1)) #Context length considered and create upper triangle matrix with 1 and 0
    
    def forward(self,x):
        b,num_tokens,d_in = x.shape

        keys = self.W_key(x) #shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        #We implicitly split the matrix by adding a 'num_heads' dimension
        #Unroll last dim: (b,num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b,num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b,num_tokens, self.num_heads,self.head_dim)
    
        #Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        #Compute scaled dot-product attention - ie. self attention with causal mask
        attn_scores = queries @ keys.transpose(2,3) # Dot product for each head

        #Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]  ##mask is been register_buffer at __init__ method, 
                                        #this creates a upper triangle matrix with True in upper triangle and False in lower

        #Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf) ##mask_bool will have only True or False in matrix, those True will be added with -inf and those False will keep the original values from attn_scores

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        #Shape: (b,num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1,2)

        #Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b,num_tokens,self.d_out)
        context_vec = self.out_proj(context_vec) #Optional projection

        return context_vec



In [ ]:
torch.manual_seed(45)

#Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],
     [0.55, 0.55, 0.79, 0.58, 0.67, 0.36],
     [0.67, 0.35, 0.09, 0.45, 0.89, 0.55]
     ]
)

batch = torch.stack((inputs,inputs),dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out=6
mha = MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads=2)  ##0.0 is dropout rate
context_vec = mha(batch)
print(context_vec)
print("Context_vec shape",context_vec.shape)

torch.Size([2, 3, 6])
tensor([[[-0.1820, -0.1442, -0.3080, -0.5791, -0.9160, -0.7152],
         [-0.1738, -0.2011, -0.3702, -0.6012, -0.8833, -0.6756],
         [-0.1439, -0.1828, -0.3438, -0.5192, -0.8283, -0.6856]],

        [[-0.1820, -0.1442, -0.3080, -0.5791, -0.9160, -0.7152],
         [-0.1738, -0.2011, -0.3702, -0.6012, -0.8833, -0.6756],
         [-0.1439, -0.1828, -0.3438, -0.5192, -0.8283, -0.6856]]],
       grad_fn=<ViewBackward0>)
Context_vec shape torch.Size([2, 3, 6])
